# Building a sample dataset for testing derivative products across different Landsat sensors

This notebook is designed to create a sample set for testing derivative products across the continent, where the testing needs to compare data derived from multiple landsat sensors. It is designed to do the following:

- Import a geojson containing the footprints of the 'golden tiles' that have been previously used for validating GeoMAD products. These tiles are distributed across Australia and across a range of environment types.
- Load a datacube dataset for the selected sensors (in this case, Landsat 7, and Landsat 8/9), using one or more of the golden tiles as the geometry for the datacube query
- Compare the resulting datasets and find the timesteps that overlap ( +/- a timeframe set by the user, e.g. 48 hours)
- If the resulting filtered dataset is very large, create a subset based on randomly selecting smaller regions or pixels
- Export the resulting geojson so it can be used as an input to test workflows

In [ ]:
import os
import datacube
import pandas as pd
import numpy as np
import xarray as xr
import geopandas as gpd
import pprint
from datetime import timedelta
import matplotlib.pyplot as plt
from datacube.utils.geometry import CRS, Geometry, GeoBox
from datacube.utils import masking
from pathlib import Path
from shapely.geometry import box

import odc.geo.xr
from odc.geo.xr import assign_crs
from odc.io.cgroups import get_cpu_quota

import sys

sys.path.insert(1, ".../Tools")
from dea_tools.datahandling import load_ard
from dea_tools.classification import collect_training_data
from dea_tools.dask import create_local_dask_cluster
from dea_tools.plotting import rgb, display_map
from dea_tools.spatial import xr_vectorize, xr_rasterize
from dea_tools.bandindices import calculate_indices

import warnings

warnings.filterwarnings("ignore")


In [ ]:
create_local_dask_cluster()


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/jenna.guffogg@ga.gov.au/proxy/41339/status,
Dashboard: /user/jenna.guffogg@ga.gov.au/proxy/41339/status,Workers: 1
Total threads: 15,Total memory: 117.21 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:39511,Workers: 1
Dashboard: /user/jenna.guffogg@ga.gov.au/proxy/41339/status,Total threads: 15
Started: Just now,Total memory: 117.21 GiB
Comm: tcp://127.0.0.1:35471,Total threads: 15
Dashboard: /user/jenna.guffogg@ga.gov.au/proxy/38819/status,Memory: 117.21 GiB
Nanny: tcp://127.0.0.1:39137,


In [ ]:
dc = datacube.Datacube()


In [ ]:
# funtion from Chad to sample ABARES landuse data


def random_sampling(
    da, n, sampling="stratified_random", manual_class_ratios=None, out_fname=None
):
    """
    Creates randomly sampled points for post-classification
    accuracy assessment.

    Params:
    -------
    da: xarray.DataArray
        A classified 2-dimensional xarray.DataArray
    n: int
        Total number of points to sample. Ignored if providing
        a dictionary of {class:numofpoints} to 'manual_class_ratios'
    sampling: str
        'stratified_random' = Create points that are randomly
        distributed within each class, where each class has a
        number of points proportional to its relative area.
        'equal_stratified_random' = Create points that are randomly
        distributed within each class, where each class has the
        same number of points.
        'random' = Create points that are randomly distributed
        throughout the image.
        'manual' = user definined, each class is allocated a
        specified number of points, supply a manual_class_ratio
        dictionary mapping number of points to each class
    manual_class_ratios: dict
        If setting sampling to 'manual', the provide a dictionary
        of type {'class': numofpoints} mapping the number of points
        to generate for each class.
    out_fname: str
        If providing a filepath name, e.g 'sample_points.shp', the
        function will export a shapefile/geojson of the sampling
        points to file.

    Output
    ------
    GeoPandas.Dataframe

    """

    if sampling not in [
        "stratified_random",
        "equal_stratified_random",
        "random",
        "manual",
    ]:
        raise ValueError(
            "Sampling strategy must be one of 'stratified_random', "
            + "'equal_stratified_random', 'random', or 'manual'"
        )

    # open the dataset as a pandas dataframe
    da = da.squeeze()
    df = da.to_dataframe(name="class")
    df = df.dropna(how="any")

    # list to store points
    samples = []

    if sampling == "stratified_random":
        # determine class ratios in image
        class_ratio = pd.DataFrame(
            {
                "proportion": df["class"].value_counts(normalize=True),
                "class": df["class"].value_counts(normalize=True).keys(),
            }
        )

        for _class in class_ratio["class"]:
            # use relative proportions of classes to sample df
            no_of_points = (
                n * class_ratio[class_ratio["class"] == _class]["proportion"].values[0]
            )
            # random sample each class
            print(
                "Class "
                + str(_class)
                + ": sampling at "
                + str(round(no_of_points))
                + " coordinates"
            )
            sample_loc = df[df["class"] == _class].sample(n=int(round(no_of_points)))
            samples.append(sample_loc)

    if sampling == "equal_stratified_random":
        classes = np.unique(df["class"])

        for _class in classes:
            # use relative proportions of classes to sample df
            no_of_points = n / len(classes)
            # random sample each classes
            try:
                sample_loc = df[df["class"] == _class].sample(
                    n=int(round(no_of_points))
                )
                print(
                    "Class "
                    + str(_class)
                    + ": sampling at "
                    + str(round(no_of_points))
                    + " coordinates"
                )
                samples.append(sample_loc)

            except ValueError:
                print(
                    "Requested more sample points than population of pixels for class "
                    + str(_class)
                    + ", skipping"
                )
                pass

    if sampling == "random":
        no_of_points = n
        # random sample entire df
        print(
            "Randomly sampling dataAraay at "
            + str(round(no_of_points))
            + " coordinates"
        )
        sample_loc = df.dropna().sample(n=int(round(no_of_points)))
        samples.append(sample_loc)

    if sampling == "manual":
        if isinstance(manual_class_ratios, dict):
            # check classes in dict match classes in data
            classes = np.unique(df["class"])
            dict_classes = list(manual_class_ratios.keys())

            if set(dict_classes).issubset([str(i) for i in classes]):
                # mask for just those classes in the provided dictionary
                mask = np.isin(classes, np.array(dict_classes).astype(type(classes[0])))
                classes = classes[mask]
                # run sampling
                for _class in classes:
                    no_of_points = manual_class_ratios.get(str(_class))
                    # random sample each class
                    try:
                        sample_loc = df[df["class"] == _class].sample(
                            n=int(round(no_of_points))
                        )
                        print(
                            "Class "
                            + str(_class)
                            + ": sampled at "
                            + str(round(no_of_points))
                            + " coordinates"
                        )
                        samples.append(sample_loc)

                    except ValueError:
                        print(
                            "Requested more sample points than population of pixels for class "
                            + str(_class)
                            + ", skipping"
                        )
                        pass

            else:
                raise ValueError(
                    "Some or all of the classes in 'manual_class_ratio' dictionary do not"
                    + " match the classes in the supplied dataArray. "
                    + "DataArray classes: "
                    + str(classes)
                    + ", Supplied dict classes: "
                    + str(list(manual_class_ratios.keys()))
                )

        else:
            raise ValueError(
                "Must supply a dictionary mapping {'class': numofpoints} if sampling"
                + " is set to 'manual'"
            )

    # join back into single datafame
    all_samples = pd.concat([samples[i] for i in range(0, len(samples))])

    # get pd.mulitindex coords as list
    y = [i[0] for i in list(all_samples.index)]
    x = [i[1] for i in list(all_samples.index)]

    # create geopandas dataframe
    gdf = gpd.GeoDataFrame(
        all_samples, crs=f"EPSG:{da.odc.crs.epsg}", geometry=gpd.points_from_xy(x, y)
    ).reset_index()

    gdf = gdf.drop(["x", "y"], axis=1)

    if out_fname is not None:
        gdf.to_file(out_fname)

    return gdf


In [ ]:
def select_tile(grid_gdf, region_code, query):
    region_code = [region_code]
    gdf = grid_gdf[grid_gdf["region_code"].isin(region_code)]
    polygon = gdf.geometry.iloc[0]

    geom = Geometry(geom=polygon, crs=gdf.crs)
    query.update({"geopolygon": geom})

    return query


In [ ]:
# Example: Search for a file named 'target_file.txt' in the current directory and all subdirectories

target_filename = "testing_tile_suite.geojson"
for root, dirs, files in os.walk("."):
    if target_filename in files:
        file_path = Path(root) / target_filename
        print(f"Found: {file_path.resolve()}")


Found: /home/jovyan/dev/development_notebooks_JAG/01_projects/tassel_cap_exploration/testing_tile_suite.geojson


In [ ]:
test_tiles_gdf = gpd.read_file(file_path.resolve())

# Extract the list of region_codes
region_codes = test_tiles_gdf["region_code"].tolist()

# remove shortlist once testing is done
region_codes = "x57y30"
# region_codes = region_codes[1:2]
pprint.pprint(region_codes)


'x57y30'


### Analysis parameters


In [ ]:
time = ("2020-02", "2020-03")
resolution = (-30, 30)
output_crs = "EPSG:3577"


In [ ]:
# resampling to larger pixels to get rid of small patches of classes. But may need to consider buffering sample selection away from edges to make sure sampled pixels are not mis-classified due to resampling effects.
query_abares = {
    "resolution": resolution,
    "output_crs": output_crs,
    "group_by": "solar_day",
    "product": "abares_clum_2023",
    "resolution": (-30, 30),
    # "resampling": "cubic",
}

query_abares = select_tile(test_tiles_gdf, region_codes, query_abares)

print(query_abares)

ds_clum = dc.load(**query_abares)


{'resolution': (-30, 30), 'output_crs': 'EPSG:3577', 'group_by': 'solar_day', 'product': 'abares_clum_2023', 'geopolygon': Geometry(POLYGON ((143.68218121959336 -35.589783511618016, 144.73746356701557 -35.507323472556806, 144.84382002673482 -36.36803490788275, 143.77983045581777 -36.45138368333508, 143.68218121959336 -35.589783511618016)), EPSG:4326)}


In [ ]:
ds_clum


<xarray.Dataset> Size: 21MB
Dimensions:      (time: 1, y: 3201, x: 3200)
Coordinates:
  * time         (time) datetime64[ns] 8B 2023-09-22T03:00:13.860935
  * y            (y) float64 26kB -3.936e+06 -3.936e+06 ... -4.032e+06
  * x            (x) float64 26kB 1.056e+06 1.056e+06 ... 1.152e+06 1.152e+06
    spatial_ref  int32 4B 3577
Data variables:
    alum_class   (time, y, x) int16 20MB 330 330 330 330 330 ... 430 430 430 430
Attributes:
    crs:           EPSG:3577
    grid_mapping:  spatial_ref

### ALUM primary classes:

TODO: refine classes to use in TC analysis so we can compare tree vs shrubs etc.

- 1: conservation and natural environments
- 2: production from relatively natural environments
- 3: production from dryland agriculture and plantations
- 4: production from irrigated agriculture and plantations
- 5: intensive uses
- 6: water (note wetlands are 651 for conservation)

In [ ]:
ds_clum_classes = (
    ds_clum // 100
) * 100  # convert all the classes to only have the parent 6 classes for now.

ds_output = ds_clum_classes.alum_class
ds_output = ds_output.squeeze().drop_vars("time")

ds_output


### Collect stratified random samples from ABARES CLUM and save out as geopackage

In [ ]:
fname_outpath_abares_samples = "abares_clum_stratified_sample_points.gpkg"

abares_sample = random_sampling(
    ds_output, 500, sampling="stratified_random", out_fname=fname_outpath_abares_samples
)

# read the geopackage file back in to then collect sample data from datacube for TC's
abares_samples_gpd = gpd.read_file(fname_outpath_abares_samples)

abares_samples_gpd.head()


Class 400: sampling at 206 coordinates
Class 300: sampling at 194 coordinates
Class 100: sampling at 32 coordinates
Class 200: sampling at 32 coordinates
Class 500: sampling at 29 coordinates
Class 600: sampling at 7 coordinates
Class 0: sampling at 0 coordinates


## Set up query and function for collecting data from datacube, stop dask client

- the `collect_sample_data` function doesn't play nicely with a local dask cluster, so the local dask client needs to be shut down before running that cell.

In [ ]:
def feature_layers(query):
    dc = datacube.Datacube()

    ds = load_ard(
        dc=dc,
        products=["ga_ls8c_ard_3", "ga_ls9c_ard_3"],
        measurements=[
            "nbart_blue",
            "nbart_green",
            "nbart_red",
            "nbart_nir",
            "nbart_swir_1",
            "nbart_swir_2",
            "oa_fmask",
            "oa_nbart_contiguity",
        ],
        cloud_mask="fmask",
        mask_pixel_quality=True,
        mask_contiguity=True,
        verbose=False,
        **query,
    )

    da = calculate_indices(
        ds, index=["TCW", "TCB", "TCG"], drop=False, collection="ga_ls_3"
    )

    return da


In [ ]:
# modify query to use bounding box of points as geometry bounds
xmin, ymin, xmax, ymax = abares_samples_gpd.total_bounds
polygon = box(xmin, ymin, xmax, ymax)

geom = Geometry(geom=polygon, crs=abares_samples_gpd.crs)


In [ ]:
# set up baseline dc query. THis will be modified by functions as needed later on.
query = {
    "time": time,
    "resolution": resolution,
    "output_crs": output_crs,
    "group_by": "solar_day",
    "geopolygon": geom,
}


In [ ]:
if get_cpu_quota() is not None:
    ncpus = round(get_cpu_quota())
else:
    ncpus = os.cpu_count()
print(f"ncpus = {ncpus}")


In [ ]:
column_names, model_input = collect_training_data(
    gdf=test_sample,
    dc_query=query,
    ncpus=ncpus,
    return_coords=True,
    field="class",
    zonal_stats="mean",  # not actually going to use this but it complains if set to False.
    feature_func=feature_layers,
)
